In [1]:
# !git clone https://github.com/myshell-ai/MeloTTS.git
# %cd MeloTTS
!pip install - e .
# !python -m unidic download


/content/MeloTTS


In [ ]:
# for running the model
!pip install torch torchaudio g2p-en gruut
# for download management
!pip install cached_path


In [11]:
del melo\text\chinese.py
del melo\text\chinese_bert.py
del melo\text\chinese_mix.py
del melo\text\japanese.py
del melo\text\japanese_bert.py
del melo\text\korean.py

# Changes in code

### add this to english.py

```python
def distribute_phone(n_phone, n_word):
    phones_per_word = [0] * n_word
    for task in range(n_phone):
        min_tasks = min(phones_per_word)
        min_index = phones_per_word.index(min_tasks)
        phones_per_word[min_index] += 1
    return phones_per_word
```

### update cleaner

```python
from . import english
from . import cleaned_text_to_sequence
import copy

language_module_map = {"EN": english}
```

### /melo/text/**init**.py

```python
def get_bert(norm_text, word2ph, language, device):
    from .english_bert import get_bert_feature as en_bert

    lang_bert_func_map = {"EN": en_bert}
    bert = lang_bert_func_map[language](norm_text, word2ph, device)
    return bert
```


In [1]:
from melo.api import TTS


c:\Users\MYSTIC Ganesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\MYSTIC Ganesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [2]:

from IPython.display import Audio

# Speed is adjustable
speed = 1.0

# CPU is sufficient for real-time inference.
# You can set it manually to 'cpu' or 'cuda' or 'cuda:0' or 'mps'
device = 'auto'  # Will automatically use GPU if available

# English
model = TTS(language='EN', device=device)
speaker_ids = model.hps.data.spk2id


None


c:\Users\MYSTIC Ganesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\utils\weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


None


In [3]:
# American accent
text = """
You will be able to reuse this secret in all of your notebooks.
"""
output_path = 'en-us.wav'
model.tts_to_file(text, speaker_ids['EN-US'], output_path, speed=speed)


# Load the wav file
audio_file_path = "en-us.wav"

# Display the audio player
Audio(audio_file_path, autoplay=True)

# 13 seconds -> cpu
# 1 seconds -> gpu


 > Text split to sentences.
You will be able to reuse this secret in all of your notebooks.
 > ===========================


c:\Users\MYSTIC Ganesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model a

# QUANTISE MODEL


In [4]:
import re
import torch
import soundfile
import numpy as np
import torch.nn as nn
import torch
from melo import utils
from melo.models import SynthesizerTrn
from melo.split_utils import split_sentence
from melo.download_utils import load_or_download_config, load_or_download_model


class TTS(nn.Module):
    def __init__(
        self, language, device="cpu", use_hf=True, config_path=None, ckpt_path=None
    ):
        super().__init__()
        device = "cpu"
        if "cuda" in device:
            assert torch.cuda.is_available()

        # config_path =
        hps = load_or_download_config(
            language, use_hf=use_hf, config_path=config_path)

        num_languages = hps.num_languages
        num_tones = hps.num_tones
        symbols = hps.symbols

        model = SynthesizerTrn(
            len(symbols),
            hps.data.filter_length // 2 + 1,
            hps.train.segment_size // hps.data.hop_length,
            n_speakers=hps.data.n_speakers,
            num_tones=num_tones,
            num_languages=num_languages,
            **hps.model,
        ).to(device)

        model.eval()
        self.model = model
        self.symbol_to_id = {s: i for i, s in enumerate(symbols)}
        self.hps = hps
        self.device = device

        # load state_dict
        checkpoint_dict = load_or_download_model(
            language, device, use_hf=use_hf, ckpt_path=ckpt_path
        )
        self.model.load_state_dict(checkpoint_dict["model"], strict=True)

        language = language.split("_")[0]

    @staticmethod
    def audio_numpy_concat(segment_data_list, sr, speed=1.0):
        audio_segments = []
        for segment_data in segment_data_list:
            audio_segments += segment_data.reshape(-1).tolist()
            audio_segments += [0] * int((sr * 0.05) / speed)
        audio_segments = np.array(audio_segments).astype(np.float32)
        return audio_segments

    @staticmethod
    def split_sentences_into_pieces(text, language, quiet=False):
        texts = split_sentence(text, language_str=language)
        if not quiet:
            print(" > Text split to sentences.")
            print("\n".join(texts))
            print(" > ===========================")
        return texts

    def tts_to_file(
        self,
        text,
        speaker_id,
        output_path: str = None,
        sdp_ratio: float = 0.2,
        noise_scale: float = 0.6,
        noise_scale_w: float = 0.8,
        speed: float = 1.0,
        quiet: bool = False,
    ):
        language = self.language
        texts = self.split_sentences_into_pieces(text, language, quiet)
        audio_list = []
        for t in texts:
            t = re.sub(r"([a-z])([A-Z])", r"\1 \2", t)
            device = self.device
            bert, ja_bert, phones, tones, lang_ids = utils.get_text_for_tts_infer(
                t, language, self.hps, device, self.symbol_to_id
            )
            with torch.no_grad():
                x_tst = phones.to(device).unsqueeze(0)
                tones = tones.to(device).unsqueeze(0)
                lang_ids = lang_ids.to(device).unsqueeze(0)
                bert = bert.to(device).unsqueeze(0)
                ja_bert = ja_bert.to(device).unsqueeze(0)
                x_tst_lengths = torch.LongTensor([phones.size(0)]).to(device)

                del phones
                speakers = torch.LongTensor([speaker_id]).to(device)

                audio = (
                    self.model.infer(
                        x_tst,
                        x_tst_lengths,
                        speakers,
                        tones,
                        lang_ids,
                        bert,
                        ja_bert,
                        sdp_ratio=sdp_ratio,
                        noise_scale=noise_scale,
                        noise_scale_w=noise_scale_w,
                        length_scale=1.0 / speed,
                    )[0][0, 0]
                    .data.cpu()
                    .float()
                    .numpy()
                )
                del x_tst, tones, lang_ids, bert, ja_bert, x_tst_lengths, speakers
                #
            audio_list.append(audio)
        torch.cuda.empty_cache()
        audio = self.audio_numpy_concat(
            audio_list, sr=self.hps.data.sampling_rate, speed=speed
        )
        soundfile.write(output_path, audio, self.hps.data.sampling_rate)


In [6]:
device = "cuda"  # Will automatically use GPU if available
model = TTS(language="EN", device=device)
speaker_ids = model.hps.data.spk2id


text_message = "no thanks"
speed = 1.0
print(text_message)

text = text_message

# American accent - save the file locally before sending it
output_path = "en-us.wav"
model.tts_to_file(text, speaker_ids["EN-US"], output_path, speed=speed)


None
None
Fuck you
 > Text split to sentences.
Fuck you
 > ===========================
